softmax converts raw logits into probabilities. substract the max for numarical stability

In [2]:
import numpy as np

def softmax(x):
    shifted = x - np.max(x, axis=-1, keepdims=True)
    exp_x = np.exp(shifted)
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

logits = np.array([2.0, 1.0, 0.1])
print(f"logits:  {logits}")
print(f"softmax: {softmax(logits)}")
print(f"sum:     {softmax(logits).sum():.4f}")

logits:  [2.  1.  0.1]
softmax: [0.65900114 0.24243297 0.09856589]
sum:     1.0000


the core function. takes q,k,v and returns the attention output plus the weighted matrix

In [3]:
def scaled_dot_product_attention(Q, K, V):
    dk = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(dk)
    weights = softmax(scores)
    output = weights @ V
    return output, weights

A full self-attention module with Wq, Wk, Wv weight matrices initialized with Xavier-like scaling.

In [4]:
class SelfAttention:
    def __init__(self, d_model, dk, dv, seed=42):
        rng = np.random.default_rng(seed)
        scale = np.sqrt(2.0 / (d_model + dk))
        self.Wq = rng.normal(0, scale, (d_model, dk))
        self.Wk = rng.normal(0, scale, (d_model, dk))
        scale_v = np.sqrt(2.0 / (d_model + dv))
        self.Wv = rng.normal(0, scale_v, (d_model, dv))
        self.dk = dk

    def forward(self, X):
        Q = X @ self.Wq
        K = X @ self.Wk
        V = X @ self.Wv
        output, weights = scaled_dot_product_attention(Q, K, V)
        return output, weights

In [5]:
sentence = ["The", "cat", "sat", "on", "the", "mat"]
n_tokens = len(sentence)
d_model = 8
dk = 4
dv = 4

rng = np.random.default_rng(42)
X = rng.normal(0, 1, (n_tokens, d_model))

attn = SelfAttention(d_model, dk, dv, seed=42)
output, weights = attn.forward(X)

print("Attention weights (each row: where that token looks):\n")
print(f"{'':>6}", end="")
for token in sentence:
    print(f"{token:>6}", end="")
print()

for i, token in enumerate(sentence):
    print(f"{token:>6}", end="")
    for j in range(n_tokens):
        w = weights[i][j]
        print(f"{w:6.3f}", end="")
    print()

Attention weights (each row: where that token looks):

         The   cat   sat    on   the   mat
   The 0.097 0.122 0.236 0.445 0.047 0.052
   cat 0.188 0.150 0.176 0.144 0.193 0.149
   sat 0.166 0.130 0.213 0.187 0.150 0.154
    on 0.172 0.144 0.146 0.115 0.208 0.215
   the 0.198 0.170 0.214 0.246 0.129 0.043
   mat 0.168 0.152 0.126 0.101 0.220 0.234


In [6]:
def ascii_heatmap(weights, tokens, chars=" ░▒▓█"):
    n = len(tokens)
    print(f"\n{'':>6}", end="")
    for t in tokens:
        print(f"{t:>6}", end="")
    print()

    for i in range(n):
        print(f"{tokens[i]:>6}", end="")
        for j in range(n):
            level = int(weights[i][j] * (len(chars) - 1) / weights.max())
            level = min(level, len(chars) - 1)
            print(f"{'  ' + chars[level] + '   '}", end="")
        print()

ascii_heatmap(weights, sentence)


         The   cat   sat    on   the   mat
   The        ░     ▒     █               
   cat  ░     ░     ░     ░     ░     ░   
   sat  ░     ░     ░     ░     ░     ░   
    on  ░     ░     ░     ░     ░     ░   
   the  ░     ░     ░     ▒     ░         
   mat  ░     ░     ░           ░     ▒   


In [7]:
import torch
import torch.nn as nn

d_model = 8
n_heads = 2
seq_len = 6

mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, batch_first=True)

X_torch = torch.randn(1, seq_len, d_model)

output, attn_weights = mha(X_torch, X_torch, X_torch)

print(f"Input shape:            {X_torch.shape}")
print(f"Output shape:           {output.shape}")
print(f"Attention weight shape: {attn_weights.shape}")
print(f"\nAttn weights (averaged over heads):")
print(attn_weights[0].detach().numpy().round(3))

Input shape:            torch.Size([1, 6, 8])
Output shape:           torch.Size([1, 6, 8])
Attention weight shape: torch.Size([1, 6, 6])

Attn weights (averaged over heads):
[[0.192 0.182 0.103 0.175 0.156 0.192]
 [0.185 0.192 0.208 0.139 0.135 0.141]
 [0.183 0.147 0.133 0.189 0.173 0.175]
 [0.136 0.211 0.167 0.119 0.16  0.208]
 [0.206 0.128 0.14  0.184 0.156 0.186]
 [0.182 0.191 0.154 0.144 0.137 0.192]]
